In [ ]:
# code has some small issue like parameter names or these kind of stuff that needs to be fixed

***reward to go***

is refinement from reinforce

“In REINFORCE, we treat the total return from the entire episode as the same signal for all actions, regardless of when those actions occurred. This causes every action to be reinforced equally — even if it happened before the relevant rewards.”

in reward to go -> its culminative reward based on steps that we are


So yes — step 0 gets the most reward if rewards are positive and accumulate, which is often the case in dense reward environments.


In [2]:
import torch
import torch.nn as nn
from torch.distributions.categorical import Categorical
from torch.optim import Adam
import numpy as np
import gymnasium as gym
from gymnasium.spaces import Discrete, Box

In [36]:
def mlp(sizes, activation=nn.Tanh, output_activation=nn.Identity):
    print(sizes)
    layers = []
    for j in range(len(sizes)-1):
        act = activation if j < len(sizes) - 2 else output_activation
        layers += [nn.Linear(sizes[0],sizes[1]),act()]

    return nn.Sequential(*layers)
        

In [9]:
#last elements get most reward in this algorithm
def reward_to_go(rews):

    n = len(rews)
    rtgs = np.zeros_like(rews)
    for i in reversed(range(n)):
        rtgs[i] = rews[i] + (rtgs[i + 1] if i+1 < n else 0)

    return rtgs

        

In [43]:


def train(env_name='CartPole-v0', hidden_sizes=[32], lr=1e-2, 
          epochs=50, batch_size=5000, render=False):

    env = gym.make('CartPole-v0')
    action_n = env.action_space.n
    obs_space = env.observation_space.shape[0]  
    model = mlp(obs_space+hidden_sizes+action_n)


    
    def policy(obs):
        logits = model(obs)
        return Categorical(logits=logits)

    def get_action(obs):
        return policy(obs).sample().item()

    def get_loss(obs,action,weights):

        logp = policy(obs).log_prob(action)     #this is log policy for action
        return -(logp * weights).mean()

    optimizer = Adam(model.parameters(),lr=lr)

    def train_one_epoch():
        # make some empty lists for logging.
        batch_obs = []          # for observations
        batch_acts = []         # for actions
        batch_weights = []      # for reward-to-go weighting in policy gradient
        batch_rets = []         # for measuring episode returns
        batch_lens = []         # for measuring episode lengths

        # reset episode-specific variables
        obs,_ = env.reset()       # first obs comes from starting distribution
        done = False            # signal from environment that episode is over
        ep_rews = []            # list for rewards accrued throughout ep

        # render first episode of each epoch
        finished_rendering_this_epoch = False

        while True:

            action = get_action(obs)

            obs,reward,done, _,_ = env.step(action)

            batch_acts.append(act)
            ep_rews.append(rew)

            if done:

                ep_ret, ep_len = sum(ep_rews), len(ep_rews)
                batch_rets.append(ep_ret)
                batch_lens.append(ep_len)

                batch_weights += list(reward_to_go(ep_rews))


                obs,_, done, ep_rews = env.reset(), False, []

                if len(batch_obs) > batch_size:
                    break
        


        optimizer.zero_grad()

        batch_loss = get_loss(
            obs = torch.FloatTensor(batch_obs) ,
            action=torch.FloatTensor(batch_acts),
            weights = torch.FloatTensor(batch_weights)
        )

        batch_loss.backward()

        optimizer.step()
    for i in range(epochs):
        batch_loss, batch_rets, batch_lens = train_one_epoch()
        print('epoch: %3d \t loss: %.3f \t return: %.3f \t ep_len: %.3f'%
                (i, batch_loss, np.mean(batch_rets), np.mean(batch_lens)))
        
    